In [27]:
# !pip install imbalanced-learn

In [28]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder as OneHot
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
# import xgboost as xgb
from sklearn.model_selection import cross_val_score,GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
data = pd.read_csv("/kaggle/input/titanic/train.csv",index_col=0)
data.info()

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv
<class 'pandas.core.frame.DataFrame'>
Index: 891 entries, 1 to 891
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Name      891 non-null    object 
 3   Sex       891 non-null    object 
 4   Age       714 non-null    float64
 5   SibSp     891 non-null    int64  
 6   Parch     891 non-null    int64  
 7   Ticket    891 non-null    object 
 8   Fare      891 non-null    float64
 9   Cabin     204 non-null    object 
 10  Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 83.5+ KB


In [29]:
# X/y
X = data.drop("Survived",axis=1)
y = data.pop("Survived")

# train/test
X_train, X_valid, y_train, y_valid = train_test_split(X,y,train_size=0.8,random_state=0)

In [30]:
# num/cat
type_df = X_train.dtypes.astype(str)
cat_col = list(type_df[type_df.isin(["object","int64"])].index)
num_col = list(set(type_df.index) - set(cat_col))

# missing value
X_train_null = X_train.isnull().sum()
null_cols = X_train_null[X_train_null > 0]
print(null_cols)
display(X_train.mask(X_train.isnull()))

Age         141
Cabin       549
Embarked      2
dtype: int64


,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,
141,3,"Boulos, Mrs. Joseph (Sultana)",female,NaN,0,2,2678,15.2458,NaN,C
440,2,"Kvillner, Mr. Johan Henrik Johannesson",male,31.0,0,0,C.A. 18723,10.5000,NaN,S
818,2,"Mallet, Mr. Albert",male,31.0,1,1,S.C./PARIS 2079,37.0042,NaN,C
379,3,"Betros, Mr. Tannous",male,20.0,0,0,2648,4.0125,NaN,C
492,3,"Windelov, Mr. Einar",male,21.0,0,0,SOTON/OQ 3101317,7.2500,NaN,S
...,...,...,...,...,...,...,...,...,...,...
836,1,"Compton, Miss. Sara Rebecca",female,39.0,1,1,PC 17756,83.1583,E49,C
193,3,"Andersen-Jensen, Miss. Carla Christine Nielsine",female,19.0,1,0,350046,7.8542,NaN,S
630,3,"O'Connell, Mr. Patrick D",male,NaN,0,0,334912,7.7333,NaN,Q


In [31]:
def missing_val_handler(df):
    df["age_missing"] = np.where(df.Age.isnull(),True, False)
    df.loc[df.Age.isnull(),["Age"]] = df.Age.mean()

    df.loc[df.Cabin.isnull(),["Cabin"]] = "missing"
    df.loc[df.Embarked.isnull(),["Embarked"]] = "missing"
    
    return df

# cardinality for cat var
cat_col = cat_col + ["age_missing"]
missing_val_fixed = missing_val_handler(X_train)
card_cat_df = missing_val_fixed[cat_col].nunique()
high_card_cols = card_cat_df[card_cat_df > 10]
print(high_card_cols)

for col in high_card_cols.index:
    print(missing_val_fixed[col].unique()[:10])
cat_col = list(set(cat_col) - set(["Name","Ticket"]))

Name      712
Ticket    569
Cabin     128
dtype: int64
['Boulos, Mrs. Joseph (Sultana)' 'Kvillner, Mr. Johan Henrik Johannesson'
 'Mallet, Mr. Albert' 'Betros, Mr. Tannous' 'Windelov, Mr. Einar'
 'Partner, Mr. Austen' 'Gilinski, Mr. Eliezer' 'McGovern, Miss. Mary'
 'Watson, Mr. Ennis Hastings' 'Bengtsson, Mr. John Viktor']
['2678' 'C.A. 18723' 'S.C./PARIS 2079' '2648' 'SOTON/OQ 3101317' '113043'
 '14973' '330931' '239856' '347068']
['missing' 'C124' 'B71' 'B30' 'A26' 'B57 B59 B63 B66' 'G6' 'B69' 'E68'
 'E121']


In [32]:
one_hot = OneHot(handle_unknown="ignore", sparse_output=False)
# one hot encoding
def cat_handler(df,train=True):
    df["Cabin"] = np.where(df.Cabin != "missing",df.Cabin.str.slice(start=0,stop=1),df.Cabin)
    #specify treatment to training/validation set
    if train:
        cat_df = pd.DataFrame(one_hot.fit_transform(df[cat_col]),index=df.index)
    else:
        cat_df = pd.DataFrame(one_hot.transform(df[cat_col]),index=df.index)
    cat_df.columns = one_hot.get_feature_names_out()
    df = pd.concat([df[num_col],cat_df],axis=1)
    
    return df

def pipeline(df,train):
    df = missing_val_handler(df)
    df = cat_handler(df,train=train)
    return df

In [33]:
X_train_processed = pipeline(X_train,train=True)
X_train_processed

,Age,Fare,Sex_female,Sex_male,Cabin_A,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,...,Parch_2,Parch_3,Parch_4,Parch_5,Parch_6,Embarked_C,Embarked_Q,Embarked_S,Embarked_missing,age_missing_False
PassengerId,,,,,,,,,,,,,,,,,,,,,
141,29.745184,15.2458,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
440,31.000000,10.5000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
818,31.000000,37.0042,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
379,20.000000,4.0125,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
492,21.000000,7.2500,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
836,39.000000,83.1583,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
193,19.000000,7.8542,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
630,29.745184,7.7333,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


In [34]:
# imbalanced target
sm = SMOTE(random_state=0)
X_res, y_res = sm.fit_resample(X_train_processed, y_train)

# baseline model
def model_eval(clf,X,y):
    result = cross_val_score(clf_rf, X=X, y=y, scoring="accuracy", cv=5, n_jobs=5)
    print(result.mean())

clf_rf = RandomForestClassifier(n_estimators=100)
model_eval(clf_rf,X_res,y_res)

0.8133636363636363


In [35]:
# not a lot of features so skip feature selection

# (explicit) interaction terms - not required for tree-based algorithms

# cluster labeling - feature engineering on num var
"""
"women and children first" - clustering based on age and sex

Here there could be some concerns about data leakage, 
yet this is just for demonstration purpose - for more 
advanced cv, one could build their own w/o using cross_val_score,
e.g. using np.random.choice
"""
kmeans = KMeans(n_clusters=4, random_state=0, n_init="auto")
kmeans.fit(X_res[["Age","Sex_female", "Sex_male"]])
X_res_cluster = X_res.copy()
X_res_cluster["cluster"] = kmeans.labels_

model_eval(clf_rf,X_res_cluster,y_res)

# target encoding - feature engineering on cat var - skip

# hyperparameter fine tuning
params = {
    "n_estimators": [100,200,500,700],
    "max_depth" : range(3,30),
    'min_samples_split' :range(2,7),
}

rf_ft = RandomForestClassifier()
clf_ft = GridSearchCV(rf_ft, params, cv=5)
clf_ft.fit(X_res_cluster, y_res)
print(clf_ft.best_params_)

0.822487012987013
{'max_depth': 17, 'min_samples_split': 3, 'n_estimators': 100}


In [36]:
# final eval
X_valid_processed = pipeline(X_valid,train=False)
X_valid_processed["cluster"] = kmeans.predict(X_valid_processed[["Age","Sex_female", "Sex_male"]])
y_valid_pred = clf_ft.predict(X_valid_processed)
print(accuracy_score(y_valid,y_valid_pred))

0.8324022346368715
